# 🎓 Day 1 · Session 4
# Multimodal AI
## Images · Documents · Audio · Video — Beyond the Text Box

In Sessions 1–3, we worked mainly with **text**.

In this session, we move beyond text and explore how modern AI systems can work with:

- 🖼️ Images
- 📄 Documents
- 🎙️ Audio
- 🎬 Video concepts
- 🔗 Multimodal fusion

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Explain what multimodal AI means
- Send images to an LLM and ask questions about them
- Use AI for OCR-style extraction and visual explanation
- Understand document understanding approaches
- Transcribe audio using OpenAI speech-to-text
- Design a multimodal teaching assistant
- Understand where multimodal AI is useful for faculty

## 🧭 Session Flow

1. Setup and API key loading  
2. What is multimodal AI?  
3. Vision demo using image URL  
4. Local image upload / base64 pattern  
5. OCR and diagram explanation  
6. Structured grading from an image  
7. Document understanding  
8. Audio transcription  
9. Multimodal teaching assistant design  
10. AI Mythbusters + Faculty lab

## 📦 Install Required Packages

Run only if packages are missing.

```bash
pip install openai python-dotenv pandas requests pypdf
```

For optional local audio transcription using Whisper:

```bash
pip install openai-whisper
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas requests pypdf

## 📚 Imports

In [ ]:
import os
import base64
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
from openai import OpenAI

## 🔑 Load API Key from `.env`

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

# 1️⃣ What is Multimodal AI?

Traditional LLMs worked mainly with text.

Multimodal models can work with multiple input types:

| Modality | Example Input | Example Task |
|---|---|---|
| Text | Prompt, email, article | Summarize, explain, classify |
| Image | Diagram, handwritten answer | Explain, extract text, grade |
| Document | PDF, report, syllabus | Summarize, answer questions |
| Audio | Lecture, student question | Transcribe, translate, summarize |
| Video | Lecture recording, experiment | Frame analysis, summary |

Key idea:

> Multimodal AI allows the same assistant to **read, see, hear and respond**.

# 2️⃣ Vision Demo: Ask Questions About an Image

We will send an image URL to the model and ask it to describe what it sees.

This is useful for:
- Diagram explanation
- Chart interpretation
- Whiteboard understanding
- Accessibility descriptions
- Lab image analysis

In [ ]:
def ask_with_image_url(
    image_url,
    question,
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=700
):
    """
    Sends an image URL and a text question to a multimodal model.
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_url}
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content

In [ ]:
# Public sample image from Wikimedia Commons.
# You can replace this with any public image URL.
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Fronalpstock_big.jpg/640px-Fronalpstock_big.jpg"

question = '''
Describe this image in detail.
Then write a short alt-text description that could help a visually impaired student.
'''

print(ask_with_image_url(image_url, question))

## 🔍 Observe

Ask participants:

1. Did the model only identify objects, or did it explain the scene?
2. Could this help create accessible course material?
3. What are the risks if the image is complex or low quality?

# 3️⃣ Faculty Use Case: Diagram Explanation

For faculty, one of the highest-value use cases is diagram explanation.

Examples:
- Circuit diagram
- Flowchart
- Biology diagram
- Architecture diagram
- Graph or chart
- Handwritten board content

In [ ]:
diagram_prompt = '''
Assume this image is shown to engineering students.

Please:
1. Describe what is visible
2. Identify important components
3. Explain it in simple teaching language
4. Mention what additional context would be needed for a complete technical explanation
'''

print(ask_with_image_url(image_url, diagram_prompt))

# 4️⃣ Local Image Pattern: Base64 Encoding

In real applications, users upload local images.

The model does not automatically know your local file path.

So we convert the local image into **base64** and send it as a data URL.

In [ ]:
def encode_image_to_data_url(image_path):
    """
    Converts a local image file into a base64 data URL.
    Supports png, jpg, jpeg.
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    suffix = image_path.suffix.lower().replace(".", "")

    if suffix == "jpg":
        suffix = "jpeg"

    if suffix not in ["png", "jpeg", "webp"]:
        raise ValueError("Supported formats: png, jpg, jpeg, webp")

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:image/{suffix};base64,{encoded}"

In [ ]:
# Optional local image demo.
# Put an image file in the same folder as this notebook and update the filename.

RUN_LOCAL_IMAGE_DEMO = False

if RUN_LOCAL_IMAGE_DEMO:
    local_image_path = "sample_diagram.png"
    data_url = encode_image_to_data_url(local_image_path)

    print(ask_with_image_url(
        data_url,
        "Explain this image as if teaching a first-year engineering student."
    ))
else:
    print("Skipping local image demo. Set RUN_LOCAL_IMAGE_DEMO = True and provide sample_diagram.png.")

# 5️⃣ OCR-Style Extraction

Modern multimodal models can read visible text from images.

This is useful for:
- Whiteboard photos
- Handwritten notes
- Scanned assignment pages
- Lab records
- Poster content

In [ ]:
ocr_question = '''
Extract any visible text from this image.
If there is no meaningful text, say that clearly.
Then explain what kind of image it appears to be.
'''

print(ask_with_image_url(image_url, ocr_question))

## ⚠️ Important Teaching Point

Multimodal AI can help with OCR-style extraction, but it is not perfect.

Always verify:
- handwritten answers
- marks
- formulas
- tables
- legal/medical/financial content

For high-stakes evaluation, faculty review is mandatory.

# 6️⃣ Structured Feedback from an Image

A powerful pattern is to ask the model to return structured JSON.

Example use case:
- Grade a handwritten answer
- Extract marks from a result sheet
- Evaluate a lab diagram
- Review a student design

In [ ]:
structured_image_prompt = '''
Assume this image is a student-submitted visual artifact.

Return your assessment as JSON with this schema:
{
  "description": "...",
  "visible_evidence": ["...", "..."],
  "possible_learning_value": "...",
  "limitations": ["...", "..."],
  "faculty_review_required": true
}

Do not invent details that are not visible.
'''

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": structured_image_prompt},
                {"type": "image_url", "image_url": {"url": image_url}}
            ]
        }
    ]
)

json_text = response.choices[0].message.content
print(json_text)

data = json.loads(json_text)
data

# 7️⃣ Document Understanding

There are three common ways to work with documents:

| Approach | How it works | Best For |
|---|---|---|
| Text extraction | Extract text from PDF, then ask model | Simple PDFs, low cost |
| Native PDF model | Send PDF directly to provider | Tables, layout, long docs |
| RAG | Chunk, embed, retrieve, answer | Large knowledge base |

In this notebook, we show the simple extraction approach.

In Session 5, we will build proper RAG.

In [ ]:
# Optional PDF extraction demo.
# Put a PDF in the same folder and set RUN_PDF_DEMO=True.

RUN_PDF_DEMO = False

if RUN_PDF_DEMO:
    import pypdf

    pdf_path = Path("sample_document.pdf")

    if not pdf_path.exists():
        raise FileNotFoundError("Please place sample_document.pdf in the notebook folder.")

    reader = pypdf.PdfReader(str(pdf_path))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)

    print("Characters extracted:", len(text))
    print(text[:1000])
else:
    print("Skipping PDF extraction demo. Set RUN_PDF_DEMO=True and provide sample_document.pdf.")

In [ ]:
# Demo with sample document text instead of requiring a PDF file.

sample_document_text = '''
Course: Introduction to Machine Learning
Credits: 4
Prerequisites: Python programming, basic probability and linear algebra.
Assessment:
- Internal Assessment: 40 marks
- Final Examination: 60 marks
Topics:
1. Supervised Learning
2. Unsupervised Learning
3. Neural Networks
4. Model Evaluation
5. Ethics in AI
'''

document_question = '''
Based only on the course document below, answer:
1. What are the prerequisites?
2. What is the assessment split?
3. List the main topics.

Course Document:
''' + sample_document_text

print(ask_with_text := client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    messages=[
        {"role": "system", "content": "Answer only from the provided document. If not present, say not found."},
        {"role": "user", "content": document_question}
    ]
).choices[0].message.content)

## 🔍 Observe

This is not full RAG yet.

Here, we manually pasted the document into the prompt.

RAG will automate:
1. Document loading
2. Chunking
3. Embedding
4. Retrieval
5. Grounded answer generation

# 8️⃣ Audio: Speech-to-Text

Audio is useful for:
- Transcribing lectures
- Converting student voice questions to text
- Creating subtitles
- Summarizing classroom discussions
- Supporting multilingual learning

In [ ]:
# Optional audio transcription demo.
# Put an audio file in the notebook folder and set RUN_AUDIO_DEMO=True.
# Supported examples: mp3, wav, m4a

RUN_AUDIO_DEMO = False

if RUN_AUDIO_DEMO:
    audio_path = Path("sample_audio.mp3")

    if not audio_path.exists():
        raise FileNotFoundError("Please place sample_audio.mp3 in the notebook folder.")

    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    print(transcript.text)
else:
    print("Skipping audio demo. Set RUN_AUDIO_DEMO=True and provide sample_audio.mp3.")

# 9️⃣ Multimodal Teaching Assistant Design

Now think architecturally.

A multimodal teaching assistant can accept:

- Image of a diagram
- Audio question from student
- PDF syllabus or textbook page
- Text question

And produce:

- Explanation
- Summary
- Feedback
- Quiz questions
- Structured JSON

In [ ]:
multimodal_use_cases = pd.DataFrame([
    {
        "Input": "Whiteboard photo",
        "AI Task": "Extract and clean notes",
        "Faculty Benefit": "Convert classroom board work into digital notes"
    },
    {
        "Input": "Handwritten answer",
        "AI Task": "Give rubric-based preliminary feedback",
        "Faculty Benefit": "Save review time, but faculty verifies"
    },
    {
        "Input": "Research paper PDF",
        "AI Task": "Summarize key findings",
        "Faculty Benefit": "Faster literature review"
    },
    {
        "Input": "Audio question",
        "AI Task": "Transcribe and answer",
        "Faculty Benefit": "Support multilingual and spoken queries"
    },
    {
        "Input": "Circuit diagram",
        "AI Task": "Explain components and working",
        "Faculty Benefit": "Better visual teaching support"
    }
])

multimodal_use_cases

# 🔟 AI Mythbusters #4

## Myth:
> Multimodal AI is just OCR.

## Reality:
OCR reads text.  
Multimodal AI can interpret **visual context**.

Example:

OCR can read:
> R1, R2, Vout

Multimodal AI can explain:
> This appears to be a voltage divider circuit, where output voltage depends on the ratio of two resistors.

That is the difference between **reading** and **understanding**.

# 1️⃣1️⃣ AI Pulse: Capability vs Experience

Discussion:

Modern AI models are not judged only by capability.

They are judged by:
- Accuracy
- Tone
- Trustworthiness
- Safety
- Speed
- User experience
- Cost

For education, a model that is technically powerful but gives overconfident or confusing answers is not enough.

Audience question:

> If you were building an AI tutor for your students, what matters most: accuracy, tone, speed, cost, or safety?

# 1️⃣2️⃣ Faculty Lab Assignment

## Build a Multimodal Teaching Tool

### Task 1
Use an image URL related to your subject and ask the model to explain it.

### Task 2
Ask the model to generate alt-text for the image.

### Task 3
Ask for structured JSON feedback on the image.

### Task 4
Use a short course document text and ask questions from it.

### Task 5
Optional: record a 30-second audio explanation and transcribe it.

### Bonus
Design a multimodal teaching assistant for your subject.

# ✅ Key Takeaways

1. AI is no longer text-only.
2. Multimodal models can see, read, hear and explain.
3. Vision is highly useful for diagrams, whiteboards and student work.
4. Document understanding is useful, but RAG is better for larger knowledge bases.
5. Audio transcription can support lecture notes and accessibility.
6. Structured JSON output makes multimodal AI usable in applications.
7. Faculty review remains essential for grading and high-stakes use cases.